<h1 style="text-align:center">Model Training</h1>

## Section 1: Imports

In [2]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from tqdm import tqdm

import matplotlib.pyplot as plt

## Section 2: Device Setup

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


## Section 3: Generic Training Utilities

### Metrics Function

In [4]:
def calculate_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    precision = precision_score(
        y_true,
        y_pred,
        average="macro"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro"
    )

    return acc, f1, precision, recall

### Train One Epoch

In [5]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()

    running_loss = 0

    all_preds = []
    all_labels = []

    loop = tqdm(loader)

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        loop.set_postfix(loss=loss.item())

    acc, f1, precision, recall = calculate_metrics(
        all_labels,
        all_preds
    )

    return (
        running_loss / len(loader),
        acc,
        f1,
        precision,
        recall
    )

### Validation Function

In [6]:
def validate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc, f1, precision, recall = calculate_metrics(
        all_labels,
        all_preds
    )

    return (
        running_loss / len(loader),
        acc,
        f1,
        precision,
        recall
    )

### Checkpoint Function

In [7]:
def save_checkpoint(model, path):

    torch.save(model.state_dict(), path)

    print(f"Model saved to {path}")

### Plot Training Curves

In [8]:
def plot_history(history):

    plt.figure(figsize=(15,5))

    plt.subplot(1,2,1)

    plt.plot(history["train_loss"], label="Train")
    plt.plot(history["val_loss"], label="Validation")

    plt.title("Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")

    plt.legend()

    plt.subplot(1,2,2)

    plt.plot(history["train_f1"], label="Train")
    plt.plot(history["val_f1"], label="Validation")

    plt.title("F1 Score Curve")
    plt.xlabel("Epoch")
    plt.ylabel("F1 Score")

    plt.legend()

    plt.show()

## Section 4: MLP Baseline

### MLP Architecture

In [9]:
class MLPBaseline(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Flatten(),

            nn.Linear(224*224*3, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 4)
        )

    def forward(self, x):

        return self.network(x)

### Initialize MLP

In [10]:
mlp_model = MLPBaseline().to(device)

### Recall Class Weights

In [12]:
import pandas as pd
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# Load manifest
df = pd.read_csv("../data/dataset_manifest.csv")

# Encode labels again
label_map = {
    "glioma": 0,
    "meningioma": 1,
    "pituitary": 2,
    "no_tumor": 3
}

df["label_encoded"] = df["label"].map(label_map)

# Compute weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["label_encoded"]),
    y=df["label_encoded"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print(class_weights)

tensor([1.0810, 1.0665, 0.9801, 0.8953])


### Loss + Optimizer

In [13]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

optimizer = optim.Adam(
    mlp_model.parameters(),
    lr=1e-3
)

### Learning Rate Scheduler

In [14]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=3,
    factor=0.5
)

### Training Loop (MLP)

In [15]:
EPOCHS = 10

best_f1 = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_f1": [],
    "val_f1": []
}

for epoch in range(EPOCHS):

    train_metrics = train_one_epoch(
        mlp_model,
        train_loader,
        criterion,
        optimizer
    )

    val_metrics = validate(
        mlp_model,
        val_loader,
        criterion
    )

    train_loss, train_acc, train_f1, _, _ = train_metrics
    val_loss, val_acc, val_f1, _, _ = val_metrics

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    scheduler.step(val_f1)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    print(f"Train F1: {train_f1:.4f}")
    print(f"Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:

        best_f1 = val_f1

        save_checkpoint(
            mlp_model,
            "../models/mlp_baseline.pth"
        )

NameError: name 'train_loader' is not defined